In [9]:
#imports
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent)) 

import pandas as pd
import matplotlib.pyplot as plt
from data.input import load_month, save_bronze


- **DATDEP** — Datum van vertrek  
- **TRAIN_NO** — Treinnummer  
- **RELATION** — Relatie  
- **TRAIN_SERV** — Spoorwegoperatoren  
- **PTCAR_NO** — Nummer van meetpunt  
- **LINE_NO_DEP** — Spoorlijn van vertrek  
- **REAL_TIME_ARR** — Uur van reële aankomst  
- **REAL_TIME_DEP** — Uur van reële vertrek  
- **PLANNED_TIME_ARR** — Uur van geplande aankomst  
- **PLANNED_TIME_DEP** — Uur van geplande vertrek  
- **DELAY_ARR** — Vertraging bij aankomst  
- **DELAY_DEP** — Vertraging bij vertrek  
- **RELATION_DIRECTION** — Richting van de relatie  
- **PTCAR_LG_NM_NL** — Naam van de halte  
- **LINE_NO_ARR** — Spoorlijn van aankomst  
- **PLANNED_DATE_ARR** — Datum van geplande aankomst  
- **PLANNED_DATE_DEP** — Datum van geplande vertrek  
- **REAL_DATE_ARR** — Datum van reële aankomst  
- **REAL_DATE_DEP** — Datum van reële vertrek  

In [10]:
# Test on one month
df = load_month('202503')
df.head()

,DATDEP,RELATION_DIRECTION,TRAIN_NO,PLANNED_DEPARTURE,PLANNED_ARRIVAL,REAL_DEPARTURE,REAL_ARRIVAL,LINE_NO_DEP,SOURCE,TARGET
0,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:23:00,2025-03-03 21:26:00,2025-03-03 21:22:40,2025-03-03 21:26:44,36N,SCHAARBEEK,BRUSSEL-NOORD
1,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:28:00,2025-03-03 21:26:00,2025-03-03 21:28:25,2025-03-03 21:26:44,36N,SCHAARBEEK,SCHAARBEEK
2,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:28:00,2025-03-03 21:30:00,2025-03-03 21:28:25,2025-03-03 21:30:24,0/2,BRUSSEL-NOORD,BRUSSEL-CONGRES
3,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:30:00,2025-03-03 21:30:00,2025-03-03 21:30:24,2025-03-03 21:30:24,0/2,BRUSSEL-NOORD,BRUSSEL-NOORD
4,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:30:00,2025-03-03 21:31:00,2025-03-03 21:30:24,2025-03-03 21:31:33,0/2,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL


In [11]:
# Basic info
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nPeriods: {df['DATDEP'].min()} → {df['DATDEP'].max()}")

Rows: 204021
Columns: ['DATDEP', 'RELATION_DIRECTION', 'TRAIN_NO', 'PLANNED_DEPARTURE', 'PLANNED_ARRIVAL', 'REAL_DEPARTURE', 'REAL_ARRIVAL', 'LINE_NO_DEP', 'SOURCE', 'TARGET']

Missing values:
DATDEP                0
RELATION_DIRECTION    0
TRAIN_NO              0
PLANNED_DEPARTURE     0
PLANNED_ARRIVAL       0
REAL_DEPARTURE        0
REAL_ARRIVAL          0
LINE_NO_DEP           0
SOURCE                0
TARGET                0
dtype: int64

Periods: 2025-03-03 00:00:00 → 2025-03-31 00:00:00


In [12]:
# validate edge-orientation for one train
train = df[df['TRAIN_NO'] == df['TRAIN_NO'].iloc[0]]
train[['TRAIN_NO', 'SOURCE', 'TARGET', 'PLANNED_DEPARTURE', 'PLANNED_ARRIVAL']]

,TRAIN_NO,SOURCE,TARGET,PLANNED_DEPARTURE,PLANNED_ARRIVAL
0,10,SCHAARBEEK,BRUSSEL-NOORD,2025-03-03 21:23:00,2025-03-03 21:26:00
1,10,SCHAARBEEK,SCHAARBEEK,2025-03-03 21:28:00,2025-03-03 21:26:00
2,10,BRUSSEL-NOORD,BRUSSEL-CONGRES,2025-03-03 21:28:00,2025-03-03 21:30:00
3,10,BRUSSEL-NOORD,BRUSSEL-NOORD,2025-03-03 21:30:00,2025-03-03 21:30:00
4,10,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL,2025-03-03 21:30:00,2025-03-03 21:31:00
...,...,...,...,...,...
199014,10,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL,2025-03-31 21:30:00,2025-03-31 21:31:00
199015,10,BRUSSEL-CONGRES,BRUSSEL-CONGRES,2025-03-31 21:31:00,2025-03-31 21:31:00
199016,10,BRUSSEL-CENTRAAL,BRUSSEL-KAPELLEKERK,2025-03-31 21:31:00,2025-03-31 21:33:00
199017,10,BRUSSEL-CENTRAAL,BRUSSEL-CENTRAAL,2025-03-31 21:33:00,2025-03-31 21:33:00


In [13]:
# validate dwell-segments
dwells = df[df['SOURCE'] == df['TARGET']]
print(f"Dwell-segments: {len(dwells)}")
print(f"Normal segments: {len(df) - len(dwells)}")
dwells.head()

Dwell-segments: 100455
Normal segments: 103566


,DATDEP,RELATION_DIRECTION,TRAIN_NO,PLANNED_DEPARTURE,PLANNED_ARRIVAL,REAL_DEPARTURE,REAL_ARRIVAL,LINE_NO_DEP,SOURCE,TARGET
1,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:28:00,2025-03-03 21:26:00,2025-03-03 21:28:25,2025-03-03 21:26:44,36N,SCHAARBEEK,SCHAARBEEK
3,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:30:00,2025-03-03 21:30:00,2025-03-03 21:30:24,2025-03-03 21:30:24,0/2,BRUSSEL-NOORD,BRUSSEL-NOORD
5,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:31:00,2025-03-03 21:31:00,2025-03-03 21:31:33,2025-03-03 21:31:33,0/2,BRUSSEL-CONGRES,BRUSSEL-CONGRES
7,2025-03-03,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,10,2025-03-03 21:33:00,2025-03-03 21:33:00,2025-03-03 21:32:24,2025-03-03 21:32:24,0/2,BRUSSEL-CENTRAAL,BRUSSEL-CENTRAAL
10,2025-03-03,ICE: BRUSSEL-ZUID -> FRANKFURT(MAIN) HBF,11,2025-03-03 06:25:00,2025-03-03 06:25:00,2025-03-03 06:25:10,2025-03-03 06:25:10,0/1,BRUSSEL-ZUID,BRUSSEL-ZUID


In [14]:
# Process all months and save to bronze
months = [
    '202503', '202504', '202506', '202507', '202508', '202509', '202510', '202511', '202512',
    '202601', '202602'
]
for month in months:
    save_bronze(month)

Opgeslagen: 202503.parquet (204021 rijen)
Opgeslagen: 202504.parquet (222373 rijen)
Opgeslagen: 202506.parquet (237655 rijen)
Opgeslagen: 202507.parquet (250645 rijen)
Opgeslagen: 202508.parquet (229380 rijen)
Opgeslagen: 202509.parquet (257119 rijen)
Opgeslagen: 202510.parquet (260735 rijen)
Opgeslagen: 202511.parquet (209889 rijen)
Opgeslagen: 202512.parquet (266687 rijen)
Opgeslagen: 202601.parquet (229195 rijen)
Opgeslagen: 202602.parquet (238487 rijen)
